# 모듈 04: 대화형 챗봇(Conversational Chatbot)

모듈 03에서 조건부 흐름과 라우터 함수로 실행 경로를 분기하는 방법을 익혔습니다.
이번에는 메시지 기록을 자동으로 누적하는 MessagesState와 LLM 호출 노드를 결합하여
대화형 챗봇 그래프를 만듭니다.

## 이 노트북에서 할 것
| 단계 | 내용 |
|------|------|
| 1 | MessagesState vs TypedDict 비교 (내장 리듀서 이해) |
| 2 | add_messages 리듀서 동작 직접 확인 |
| 3 | LLM 호출 노드 + 단일 챗봇 그래프 구현 |
| 4 | 멀티턴 대화 구현 (이전 상태 전달 패턴) |
| 5 | MessagesState 확장 (추가 필드 사용) |

예상 소요 시간: 약 35분  
전제 조건: 모듈 03 완료 (조건부 엣지, 라우터 함수)

## 학습 목표

---

1. `MessagesState`가 `messages` 필드에 `add_messages` 리듀서를 내장한 이유와
   일반 TypedDict와의 차이를 설명할 수 있다
2. `llm.invoke(state["messages"])` → `return {"messages": [response]}` 패턴으로
   LLM 호출 노드를 작성할 수 있다
3. 이전 호출의 최종 상태를 다음 호출에 전달하는 멀티턴 대화 패턴을 구현할 수 있다

## 전체 흐름

---

```
┌──────────────────────────────────────────────────────┐
│                                                      │
│  [1] MessagesState vs TypedDict 비교                 │
│       ↓                                              │
│  [2] add_messages 리듀서 동작 확인                   │
│       ↓                                              │
│  [3] 환경 설정 + LLM 초기화                          │
│       ↓                                              │
│  [4] 단일 응답 챗봇 그래프 구현                      │
│       ↓                                              │
│  [5] 멀티턴 대화 구현                                │
│       ↓                                              │
│  [6] MessagesState 확장 + 메시지 기록 분석           │
│                                                      │
└──────────────────────────────────────────────────────┘
```

## 섹션 1: MessagesState vs TypedDict

---

TypedDict로 메시지 리스트를 직접 관리하면 덮어쓰기 문제가 발생한다:

```python
# TypedDict + 직접 리스트: 문제
class State(TypedDict):
    messages: list   # 노드가 {"messages": [...]} 반환하면 덮어쓰기

# MessagesState: 해결
from langgraph.graph import MessagesState
# messages 필드가 add_messages 리듀서로 자동 누적됨
```

| 구분 | 동작 | 사용 시점 |
|------|------|----------|
| `TypedDict + list` | 반환할 때마다 덮어쓰기 | 대화 기록 불필요 |
| `MessagesState` | 반환한 메시지를 기존 목록에 **추가** | 대화 기록 누적 필요 |

MessagesState 내부 구조:

```python
# langgraph 내부 정의 (참고용)
class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
```

In [ ]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import MessagesState, StateGraph, END
from langgraph.graph.message import add_messages

notebook_dir = os.path.dirname(os.path.abspath('__file__'))
langgraph_root = os.path.dirname(notebook_dir)
project_root = os.path.dirname(langgraph_root)
langchain_env = os.path.join(project_root, 'LangChain', '.env')
langgraph_env = os.path.join(langgraph_root, '.env')

if os.path.exists(langchain_env):
    load_dotenv(langchain_env)
    print('[OK] LangChain/.env 로드 완료')
elif os.path.exists(langgraph_env):
    load_dotenv(langgraph_env)
    print('[OK] LangGraph/.env 로드 완료')
else:
    print('[FAIL] .env 파일을 찾을 수 없습니다')

api_key = os.getenv('GOOGLE_API_KEY')
print(f'[OK] GOOGLE_API_KEY: {"설정됨" if api_key else "[FAIL] 미설정"}')

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.7)
print('[OK] ChatGoogleGenerativeAI 초기화 완료 (gemini-2.0-flash)')

## 섹션 2: add_messages 리듀서 동작

---

`add_messages`는 메시지를 덮어쓰지 않고 기존 목록에 추가하는 리듀서 함수다.
동작을 그래프 없이 직접 확인할 수 있다:

```python
from langgraph.graph.message import add_messages

existing = [HumanMessage(content="안녕")]
new_msg  = [AIMessage(content="반갑습니다!")]

result = add_messages(existing, new_msg)
# → [HumanMessage("안녕"), AIMessage("반갑습니다!")]
```

메시지 객체 타입:

| 클래스 | 역할 |
|--------|------|
| `HumanMessage` | 사용자가 보낸 메시지 |
| `AIMessage` | LLM이 반환한 메시지 |
| `SystemMessage` | 시스템 지시 (프롬프트) |

In [ ]:
# add_messages 동작 직접 확인 (그래프 없이)
history = []

# 첫 번째 메시지 추가
history = add_messages(history, [HumanMessage(content="파이썬이 뭐야?")])
print(f'메시지 수: {len(history)}')

# 두 번째 메시지 추가
history = add_messages(history, [AIMessage(content="파이썬은 프로그래밍 언어입니다.")])
print(f'메시지 수: {len(history)}')

# 세 번째 메시지 추가
history = add_messages(history, [HumanMessage(content="알려줘서 고마워")])
print(f'메시지 수: {len(history)}')

print()
print('=== 누적된 메시지 기록 ===')
for msg in history:
    role = '사용자' if isinstance(msg, HumanMessage) else 'AI'
    print(f'  [{role}] {msg.content}')

print()
print('[OK] add_messages는 덮어쓰지 않고 누적합니다')

## 섹션 3: LLM 호출 노드 패턴

---

챗봇 노드는 세 줄 패턴이다:

```python
def chatbot(state: MessagesState) -> dict:
    response = llm.invoke(state["messages"])   # 전체 대화 기록 전달
    return {"messages": [response]}            # AI 응답 하나만 반환
```

그래프 구조:

```
[START] → [chatbot] → [END]
```

`llm.invoke(messages)`는 대화 기록 전체를 받아 다음 응답을 반환한다.
반환된 `AIMessage` 객체가 `add_messages` 리듀서에 의해 기존 기록에 추가된다.

In [ ]:
def chatbot(state: MessagesState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

chat_graph = StateGraph(MessagesState)
chat_graph.add_node("chatbot", chatbot)
chat_graph.set_entry_point("chatbot")
chat_graph.add_edge("chatbot", END)
chat_app = chat_graph.compile()

print('=== 단일 응답 챗봇 ===')
result = chat_app.invoke({
    "messages": [HumanMessage(content="LangGraph가 무엇인지 한 문장으로 설명해줘")]
})
print(f'사용자: LangGraph가 무엇인지 한 문장으로 설명해줘')
print(f'AI: {result["messages"][-1].content}')
print(f'총 메시지 수: {len(result["messages"])}')

## 섹션 4: 멀티턴 대화 패턴

---

멀티턴 대화는 **이전 호출의 최종 상태를 다음 호출의 초기 상태로 전달**하는 패턴이다:

```python
# 1턴
state = app.invoke({"messages": [HumanMessage(content="내 이름은 철수야")]})

# 2턴: 이전 메시지 기록 + 새 메시지
state = app.invoke({
    "messages": state["messages"] + [HumanMessage(content="내 이름이 뭐야?")]
})
# → LLM이 이전 대화 기록을 보고 "철수"라고 답함
```

대화 기록이 전달되는 구조:

```
1턴: [Human: "철수야"] → LLM → [Human: "철수야", AI: "안녕하세요!"]
                                        ↓
2턴: [Human: "철수야", AI: "...", Human: "이름 뭐야?"] → LLM → ...
```

In [ ]:
print('=== 멀티턴 대화 (3턴) ===')

# 1턴
state = chat_app.invoke({
    "messages": [HumanMessage(content="내 이름은 민준이야. 만나서 반가워.")]
})
print(f'[1턴] 사용자: 내 이름은 민준이야. 만나서 반가워.')
print(f'[1턴] AI: {state["messages"][-1].content[:80]}...')

# 2턴
state = chat_app.invoke({
    "messages": state["messages"] + [HumanMessage(content="내 이름이 뭐야?")]
})
print(f'[2턴] 사용자: 내 이름이 뭐야?')
print(f'[2턴] AI: {state["messages"][-1].content}')

# 3턴
state = chat_app.invoke({
    "messages": state["messages"] + [HumanMessage(content="지금까지 몇 번 대화했어?")]
})
print(f'[3턴] 사용자: 지금까지 몇 번 대화했어?')
print(f'[3턴] AI: {state["messages"][-1].content[:80]}...')

print()
print(f'[OK] 최종 메시지 기록 총 {len(state["messages"])}개 (사용자 3 + AI 3)')

In [ ]:
# MessagesState를 상속하거나 Annotated로 직접 정의하여 필드 추가
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]
    turn_count: int

def chatbot_with_count(state: ChatState) -> dict:
    response = llm.invoke(state["messages"])
    return {
        "messages": [response],
        "turn_count": state["turn_count"] + 1,
    }

count_graph = StateGraph(ChatState)
count_graph.add_node("chatbot", chatbot_with_count)
count_graph.set_entry_point("chatbot")
count_graph.add_edge("chatbot", END)
count_app = count_graph.compile()

print('=== MessagesState 확장: turn_count 추가 ===')
state2 = {"messages": [HumanMessage(content="파이썬의 장점을 하나만 말해줘")], "turn_count": 0}
state2 = count_app.invoke(state2)
print(f'AI: {state2["messages"][-1].content[:80]}...')
print(f'턴 수: {state2["turn_count"]}')

state2 = count_app.invoke({
    "messages": state2["messages"] + [HumanMessage(content="단점도 하나 말해줘")],
    "turn_count": state2["turn_count"]
})
print(f'AI: {state2["messages"][-1].content[:80]}...')
print(f'턴 수: {state2["turn_count"]}')
print('[OK] TypedDict + Annotated로 MessagesState와 동일한 메시지 누적 동작 구현 가능')

## 섹션 5: 메시지 기록 구조 분석

---

메시지 기록 전체를 순회하면 대화 흐름을 확인할 수 있다:

```python
for msg in state["messages"]:
    role = "사용자" if isinstance(msg, HumanMessage) else "AI"
    print(f'[{role}] {msg.content}')
```

메시지 객체의 주요 속성:

| 속성 | 설명 |
|------|------|
| `.content` | 메시지 텍스트 |
| `.type` | `"human"` / `"ai"` / `"system"` |
| `isinstance(msg, HumanMessage)` | 타입 확인 |

In [ ]:
print('=== 최종 대화 기록 (state2) ===')
for i, msg in enumerate(state2["messages"], 1):
    role = '사용자' if isinstance(msg, HumanMessage) else 'AI  '
    content_preview = msg.content[:60] + '...' if len(msg.content) > 60 else msg.content
    print(f'  {i}. [{role}] {content_preview}')

print()
print(f'총 메시지 수: {len(state2["messages"])}')
print(f'총 턴 수: {state2["turn_count"]}')
print('[OK] messages 리스트에 HumanMessage와 AIMessage가 교대로 쌓입니다')

## 그래프 구조 시각화

---

`get_graph().print_ascii()`로 두 챗봇 그래프의 구조를 확인합니다.  
단순한 선형 구조이지만, 모듈 05에서는 여기에 ToolNode가 추가되어 루프 구조가 됩니다.

In [ ]:
print('=== 기본 챗봇 그래프 ===')
chat_app.get_graph().print_ascii()

print()
print('=== 확장 챗봇 그래프 (turn_count) ===')
count_app.get_graph().print_ascii()

## 마무리

---

핵심 패턴 요약:

```python
from typing import TypedDict, Annotated
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from langgraph.graph import MessagesState, StateGraph, END

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.7)

# 챗봇 노드: 3줄 패턴
def chatbot(state: MessagesState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

# 그래프 조립
graph = StateGraph(MessagesState)
graph.add_node("chatbot", chatbot)
graph.set_entry_point("chatbot")
graph.add_edge("chatbot", END)
app = graph.compile()

# 단일 호출
result = app.invoke({"messages": [HumanMessage(content="안녕")]})

# 멀티턴: 이전 messages에 새 메시지 추가
result = app.invoke({
    "messages": result["messages"] + [HumanMessage(content="다시 안녕")]
})
```

핵심 API 레퍼런스:

| API | 설명 |
|-----|------|
| `MessagesState` | `messages` 필드 + `add_messages` 리듀서 내장 |
| `add_messages(existing, new)` | 메시지 누적 리듀서 함수 |
| `HumanMessage(content="...")` | 사용자 메시지 생성 |
| `llm.invoke(messages)` | 대화 기록 전달 후 AIMessage 반환 |
| `state["messages"][-1].content` | 마지막 AI 응답 텍스트 추출 |

## 다음 모듈 예고 + 자기 점검

---

### 다음 모듈 05 - 도구를 가진 에이전트

- `@tool` 데코레이터로 도구 함수 정의
- `ToolNode`: LLM의 `tool_calls`를 받아 실제 함수를 실행하는 노드
- `tools_condition`: 도구 호출 여부를 자동 판별하는 내장 라우터
- 에이전트 루프: `agent` → (도구 호출 시) `tools` → `agent` → ... → `END`

코드 미리보기:

```python
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """두 수를 더한다"""
    return a + b

tool_node = ToolNode(tools=[add])
llm_with_tools = llm.bind_tools([add])

graph.add_conditional_edges("agent", tools_condition)
```

### 자기 점검 체크리스트

- [ ] `MessagesState`를 사용하면 메시지가 왜 덮어쓰이지 않는지 설명할 수 있나요?
- [ ] `chatbot` 노드의 3줄 패턴을 외워서 쓸 수 있나요?
- [ ] 멀티턴 대화를 구현할 때 이전 상태를 어떻게 전달하나요?